In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

omop_path = "/path/to/exports/omop_export"
output_path = "/path/to/exports/meds_output_spark"
os.makedirs(f"{output_path}/data", exist_ok=True)
os.makedirs(f"{output_path}/metadata", exist_ok=True)

CSV_READ_OPTS = {
    "header": "true",
    "escape": '"',
    "quote": '"',
    "multiLine": "true",
}

# ── 1. Build concept_id → code string map ──────────────────────────────────
concept = spark.read.csv(f"{omop_path}/concept", **CSV_READ_OPTS) \
    .select(
        F.col('concept_id').cast('long'),
        F.concat(F.col('vocabulary_id'), F.lit('/'), F.col('concept_code')).alias('code')
    )

# ── 2. Helper to map concept IDs to code strings ───────────────────────────
def map_concept(df, src_concept_col, std_concept_col):
    """Prefer standard concept, fall back to source concept.
    """
    return df \
        .join(concept.alias('src_c'), 
              F.expr(f"try_cast(`{src_concept_col}` as long)") == F.col('src_c.concept_id'), 'left') \
        .withColumnRenamed('code', 'src_code') \
        .join(concept.alias('std_c'), 
              F.expr(f"try_cast(`{std_concept_col}` as long)") == F.col('std_c.concept_id'), 'left') \
        .withColumnRenamed('code', 'std_code') \
        .withColumn('code', F.coalesce(
            F.when(F.expr(f"try_cast(`{std_concept_col}` as long)") != 0, F.col('std_code')),
            F.when(F.expr(f"try_cast(`{src_concept_col}` as long)") != 0, F.col('src_code'))
        ))

# ── 3. Standard output columns ──────────────────────────────────────────────
def base_cols(df, time_col, table_name,
              numeric_col=None, text_col=None,
              end_col=None, unit_col=None):
    df = df \
        .withColumn('subject_id', F.expr("try_cast(person_id as long)")) \
        .withColumn('time', F.expr(f"try_cast(`{time_col}` as timestamp)")) \
        .withColumn('table', F.lit(table_name)) \
        .withColumn('numeric_value', F.col(numeric_col).cast('float') if numeric_col else F.lit(None).cast('float')) \
        .withColumn('text_value', F.col(text_col).cast('string') if text_col else F.lit(None).cast('string')) \
        .withColumn('end', F.expr(f"try_cast(`{end_col}` as timestamp)") if end_col else F.lit(None).cast('timestamp')) \
        .withColumn('unit', F.col(unit_col).cast('string') if unit_col else F.lit(None).cast('string')) \
        .withColumn('visit_id', F.col('visit_occurrence_id').cast('string') if 'visit_occurrence_id' in df.columns else F.lit(None).cast('string'))
    return df.select('subject_id', 'time', 'code',
                     'numeric_value', 'text_value',
                     'end', 'unit', 'table', 'visit_id') \
             .filter(F.col('code').isNotNull()) \
             .filter(F.col('time').isNotNull()) \
             .filter(F.col('subject_id').isNotNull())

# ── 4. Process each OMOP table ──────────────────────────────────────────────

all_events = []

# PERSON → birth, gender, race, ethnicity
person = spark.read.csv(f"{omop_path}/person", **CSV_READ_OPTS)
# Also fix the birth time casting
#
# FIX: femr 0.2.2/0.2.3 (used by StanfordShahLab/clmbr-t-base) hard-pins
# meds==0.1.3, where meds.birth_code == "SNOMED/184099003" (the modern
# "MEDS_BIRTH" sentinel wasn't introduced until meds 0.3.0). femr's
# get_patient_birthdate() does an exact string match against that constant,
# and CLMBR-t-base's own vocabulary was built around it -- confirmed both by
# femr's official usage example and by tokenizer.code_lookup containing
# 'SNOMED/184099003' but not 'MEDS_BIRTH'.
birth = person \
    .withColumn('code', F.lit('SNOMED/184099003')) \
    .withColumn('subject_id', F.col('person_id').cast('long')) \
    .withColumn('time', F.coalesce(
        F.expr("try_cast(birth_datetime as timestamp)"),
        F.expr("try_cast(concat_ws('-', year_of_birth, coalesce(month_of_birth, '1'), coalesce(day_of_birth, '1')) as timestamp)")
    )) \
    .withColumn('numeric_value', F.lit(None).cast('float')) \
    .withColumn('text_value', F.lit(None).cast('string')) \
    .withColumn('end', F.lit(None).cast('timestamp')) \
    .withColumn('unit', F.lit(None).cast('string')) \
    .withColumn('visit_id', F.lit(None).cast('string')) \
    .withColumn('table', F.lit('person')) \
    .select('subject_id', 'time', 'code', 'numeric_value', 
            'text_value', 'end', 'unit', 'table', 'visit_id')
all_events.append(birth)

for concept_field in ['gender_concept_id', 'race_concept_id', 'ethnicity_concept_id']:
    mapped = person \
        .join(concept, F.col(concept_field).cast('long') == concept.concept_id, 'left') \
        .withColumn('subject_id', F.col('person_id').cast('long')) \
        .withColumn('time', F.coalesce(
            F.to_timestamp('birth_datetime'),
            F.to_timestamp(F.concat_ws('-', 'year_of_birth',
                           F.coalesce('month_of_birth', F.lit('1')),
                           F.coalesce('day_of_birth', F.lit('1'))))
        )) \
        .withColumn('numeric_value', F.lit(None).cast('float')) \
        .withColumn('text_value', F.lit(None).cast('string')) \
        .withColumn('end', F.lit(None).cast('timestamp')) \
        .withColumn('unit', F.lit(None).cast('string')) \
        .withColumn('visit_id', F.lit(None).cast('string')) \
        .withColumn('table', F.lit('person')) \
        .filter(F.col('code').isNotNull()) \
        .select('subject_id', 'time', 'code', 'numeric_value',
                'text_value', 'end', 'unit', 'table', 'visit_id')
    all_events.append(mapped)

print("Person done")

# CONDITION
condition = spark.read.csv(f"{omop_path}/condition_occurrence", **CSV_READ_OPTS)
condition = map_concept(condition, 'condition_source_concept_id', 'condition_concept_id')
condition = base_cols(condition, 'condition_start_datetime', 'condition')
all_events.append(condition)
print(f"Condition done: {condition.count()} rows")

# MEASUREMENT
MEASUREMENT_CODE_REMAP = {
    'SNOMED/60621009': 'SNOMED/698094009',   # BMI
    'LOINC/8867-4': 'SNOMED/364075005',  # heart rate
    'CPT4/87491': 'LOINC/21613-5', # Chlamydia trachomatis
    'CPT4/87591': 'LOINC/24111-7', # Neisseria gonorrhoeae
    'LOINC/8480-6': 'SNOMED/72313002' # systolic BP
}

MEASUREMENT_PLAIN_CODE_OVERRIDE = [
    'LOINC/20578-1', 'LOINC/49752-9', 'LOINC/34376-4', 'LOINC/32133-1',
    'LOINC/12610-2', 'LOINC/18185-9', 'LOINC/1547-9', 'LOINC/71693-6',
] # categorical variables that would be dropped bc they aren't clean numerical values, just scraping the values so the code stays in pt medical history

measurement = spark.read.csv(f"{omop_path}/measurement", **CSV_READ_OPTS)
measurement = map_concept(measurement, 'measurement_source_concept_id', 'measurement_concept_id')

remap_expr = F.col('code')
for old_code, new_code in MEASUREMENT_CODE_REMAP.items():
    remap_expr = F.when(F.col('code') == old_code, F.lit(new_code)).otherwise(remap_expr)
measurement = measurement.withColumn('code', remap_expr)

# resolve unit_concept_id -> standardized VOCAB/CODE string, with unit_source_value
# as fallback. NOTE: confirmed via direct tokenizer test that `unit` has no effect
# on this model's tokenization (tokens are identical regardless of unit value) --
# this change is purely for MEDS-output hygiene / future-proofing, not for any
# current model-performance benefit.
unit_concept_df = spark.read.csv(f"{omop_path}/concept", **CSV_READ_OPTS) \
    .select(
        F.col('concept_id').alias('unit_concept_id'),
        F.concat_ws('/', F.col('vocabulary_id'), F.col('concept_code')).alias('unit_concept_code')
    )
measurement = measurement.join(unit_concept_df, on='unit_concept_id', how='left')

measurement = measurement.withColumn(
    'resolved_unit',
    F.coalesce(F.col('unit_concept_code'), F.col('unit_source_value'))
)

measurement = base_cols(measurement, 'measurement_datetime', 'measurement',
                        numeric_col='value_as_number',
                        text_col='value_source_value',
                        unit_col='resolved_unit')

measurement = measurement.withColumn(
    'numeric_value',
    F.when(F.col('code').isin(MEASUREMENT_PLAIN_CODE_OVERRIDE), F.lit(None).cast('float'))
     .otherwise(F.col('numeric_value'))
)

all_events.append(measurement)
print(f"Measurement done: {measurement.count()} rows")

# OBSERVATION
observation = spark.read.csv(f"{omop_path}/observation", **CSV_READ_OPTS)
observation = map_concept(observation, 'observation_source_concept_id', 'observation_concept_id')
observation = base_cols(observation, 'observation_datetime', 'observation',
                        numeric_col='value_as_number',
                        text_col='value_as_string',
                        unit_col='unit_source_value')
all_events.append(observation)
print(f"Observation done: {observation.count()} rows")

# DRUG EXPOSURE
drug = spark.read.csv(f"{omop_path}/drug_exposure", **CSV_READ_OPTS)
drug = map_concept(drug, 'drug_source_concept_id', 'drug_concept_id')
drug = base_cols(drug, 'drug_exposure_start_datetime', 'drug_exposure')
all_events.append(drug)
print(f"Drug done: {drug.count()} rows")

# PROCEDURE
procedure = spark.read.csv(f"{omop_path}/procedure_occurrence", **CSV_READ_OPTS)
procedure = map_concept(procedure, 'procedure_source_concept_id', 'procedure_concept_id')
procedure = base_cols(procedure, 'procedure_datetime', 'procedure')
all_events.append(procedure)
print(f"Procedure done: {procedure.count()} rows")

# VISIT
visit = spark.read.csv(f"{omop_path}/visit_occurrence", **CSV_READ_OPTS)
visit = map_concept(visit, 'visit_source_concept_id', 'visit_concept_id')
visit = base_cols(visit, 'visit_start_datetime', 'visit',
                  end_col='visit_end_datetime')
all_events.append(visit)
print(f"Visit done: {visit.count()} rows")

# DEVICE EXPOSURE
device = spark.read.csv(f"{omop_path}/device_exposure", **CSV_READ_OPTS)
device = map_concept(device, 'device_source_concept_id', 'device_concept_id')
device = base_cols(device, 'device_exposure_start_datetime', 'device_exposure')
all_events.append(device)
print(f"Device done: {device.count()} rows")

# DEATH
death = spark.read.csv(f"{omop_path}/death", **CSV_READ_OPTS)
death = death \
    .withColumn('code', F.lit('MEDS_DEATH')) \
    .withColumn('subject_id', F.col('person_id').cast('long')) \
    .withColumn('time', F.expr("try_cast(death_datetime as timestamp)")) \
    .withColumn('numeric_value', F.lit(None).cast('float')) \
    .withColumn('text_value', F.lit(None).cast('string')) \
    .withColumn('end', F.lit(None).cast('timestamp')) \
    .withColumn('unit', F.lit(None).cast('string')) \
    .withColumn('visit_id', F.lit(None).cast('string')) \
    .withColumn('table', F.lit('death')) \
    .filter(F.col('time').isNotNull()) \
    .select('subject_id', 'time', 'code', 'numeric_value',
            'text_value', 'end', 'unit', 'table', 'visit_id')
all_events.append(death)
print(f"Death done: {death.count()} rows")

# ── 5. Combine and write ────────────────────────────────────────────────────
print("Combining all events...")
all_df = all_events[0]
for df in all_events[1:]:
    all_df = all_df.union(df)

# Sort by subject_id, time, code
all_df = all_df.orderBy('subject_id', 'time', 'code')

print(f"Total events: {all_df.count()}")
print("Writing to parquet...")

# Write in 100 shards
all_df.repartition(100, 'subject_id') \
      .write.parquet(f"{output_path}/data", mode='overwrite')

print("Done! Writing metadata...")

# Write minimal metadata
import json
metadata = {
    "dataset_name": "",
    "dataset_version": "",
    "etl_name": "spark_omop_etl",
    "etl_version": "1.0",
    "meds_version": "0.3.3"
}
with open(f"{output_path}/metadata/dataset.json", 'w') as f:
    json.dump(metadata, f)

print(f"Complete! Output at {output_path}")